# Music Review Insight Finder: Learning Rate and Training Length

Experiments C and D tested two changes to the class-weighted model from
Experiment B. C used a smaller learning rate. D tested more training epochs at
that lower rate. Both started fresh rather than continuing an earlier model.

The experiments used Qwen2.5-0.5B-Instruct with LoRA, which trained a small set
of added parameters while keeping the pretrained model's weights unchanged.
Each run used the same 140 training passages and 33 validation passages.
The dataset uses AI-assisted labels with human review.

The saved outputs document both completed experiments. B retained the highest
validation macro-F1, with a very small margin over D.

## 1. Training Environment

The experiments ran in Google Colab with a Tesla T4 GPU, Python 3.12.13, and
PyTorch 2.11.0+cu128. The dependency checks kept the same package versions as
Experiment B.

For reproduction, use the matching runtime and run the setup checks first.
If the setup cell installs packages, restart the session before loading the
input bundle. Run the sections in order rather than using Run All.

In [1]:
import os, sys
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["WANDB_DISABLED"] = "true"
assert (3,10) <= sys.version_info[:2] <= (3,12), "Choose runtime version 2026.07 / Python 3.12."
import torch
assert torch.cuda.is_available(), "Select a free T4 GPU first."
assert "T4" in torch.cuda.get_device_name(0), "This bounded draft uses the known working T4 path."
assert torch.__version__.split("+")[0] == "2.11.0", "Use runtime version 2026.07 for the known working PyTorch build."
assert torch.cuda.get_device_properties(0).total_memory >= 7*1024**3
print(sys.version.split()[0], torch.__version__, torch.cuda.get_device_name(0))

3.12.13 2.11.0+cu128 Tesla T4


In [2]:
import importlib.metadata, subprocess
PINS = {"transformers":"4.57.1", "peft":"0.17.1", "accelerate":"1.10.1", "safetensors":"0.6.2",
        "numpy":"1.26.4", "pandas":"2.2.3", "matplotlib":"3.9.4", "scikit-learn":"1.6.1"}
def installed_version(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return None
needs_install = any(installed_version(name) != version for name,version in PINS.items())
SETUP_NEEDS_RESTART = False
if needs_install:
    subprocess.check_call([sys.executable,"-m","pip","install","--quiet"] + [name+"=="+v for name,v in PINS.items()])
    SETUP_NEEDS_RESTART = True
    print("Packages installed. Runtime > Restart session NOW, then resume at section 2. Do not Run all.")
else:
    print("Pinned dependencies already match. Continue at section 2; no installation needed.")

Packages installed. Runtime > Restart session NOW, then resume at section 2. Do not Run all.


## 2. Training and Validation Data

The input file was `music_insight_TRAIN_VALIDATION_ONLY_C_lr1e4_bundle.zip`.
It contained the frozen training and validation passages, their labeling
records, and the matching helper code. It excluded the historical test set.

The checks below verified the archive and its files before loading the data.
The passages, category labels, and grouped split assignments stayed the same
across B, C, and D. Each experiment saved its results in a separate folder.

In [1]:
assert not globals().get("SETUP_NEEDS_RESTART",False), "Restart the session after installation; resume at section 2."
import sys, importlib.metadata
PINS = {"transformers":"4.57.1", "peft":"0.17.1", "accelerate":"1.10.1", "safetensors":"0.6.2",
        "numpy":"1.26.4", "pandas":"2.2.3", "matplotlib":"3.9.4", "scikit-learn":"1.6.1"}
for name,version in PINS.items(): assert importlib.metadata.version(name) == version, "Run section 1 setup first."
from google.colab import files
uploaded = files.upload()
assert len(uploaded)==1, "Select the one train/validation-only bundle."
archive_name = next(iter(uploaded))

Saving music_insight_TRAIN_VALIDATION_ONLY_C_lr1e4_bundle.zip to music_insight_TRAIN_VALIDATION_ONLY_C_lr1e4_bundle.zip


In [2]:
from pathlib import Path
import hashlib, io, json, tempfile, zipfile
EXPECTED_BUNDLE_SHA256 = 'b0b4805c88fd236a05529cc304fa4f2aa011bd18e2e3100f5702ff80a1718772'
EXPECTED_MEMBERS = ['bundle_sha256.json', 'c_contract.py', 'c_diagnostics.py', 'c_protocol.py', 'c_training.py', 'data/manifest.json', 'data/train.csv', 'data/validation.csv', 'preparation_provenance.json', 'reference_B/experiment_summary.json', 'reference_B/settings.json', 'reference_B/source.json', 'requirements_colab_validation_only.txt']
assert hashlib.sha256(uploaded[archive_name]).hexdigest() == EXPECTED_BUNDLE_SHA256, "Choose the matching C bundle."
with zipfile.ZipFile(io.BytesIO(uploaded[archive_name])) as z:
    assert len(z.namelist()) == len(EXPECTED_MEMBERS) and set(z.namelist()) == set(EXPECTED_MEMBERS)
    assert sum(i.file_size for i in z.infolist()) < 5_000_000
    blobs = {name:z.read(name) for name in EXPECTED_MEMBERS}
hashes = json.loads(blobs["bundle_sha256.json"])
assert set(hashes) == set(EXPECTED_MEMBERS)-{"bundle_sha256.json"}
for name,digest in hashes.items():
    assert hashlib.sha256(blobs[name]).hexdigest() == digest, name
assert not any(name in sys.modules for name in ["c_contract","c_protocol","c_training","c_diagnostics"]), "C helpers already loaded: preserve this attempt; use a fresh C session for an authorized retry."
C_WORK = Path(tempfile.mkdtemp(prefix="music_followup_C_",dir="/content"))
for name,blob in blobs.items():
    path=C_WORK/name; path.parent.mkdir(parents=True,exist_ok=True); path.write_bytes(blob)
sys.path.insert(0,str(C_WORK))
import c_contract as contract
import c_protocol as protocol
parts, manifest = contract.load_data(C_WORK)
protocol.check_c_request(protocol.C_NAME, protocol.C_LEARNING_RATE, manifest)
C_OUTPUT = protocol.create_output(C_WORK)
print("Verified unchanged rows:", {s:len(rows) for s,rows in parts.items()})
print("C output folder:", C_OUTPUT)

Verified unchanged rows: {'train': 140, 'validation': 33}
C output folder: /content/music_followup_C_kkipo9qx/C_results


## 3. Reference Experiment B

Experiment B used class weights to give less common categories more influence
during training. Errors on these categories received a larger penalty, with
the goal of improving performance across all three categories.

The input bundle included B's saved settings and results from the completed
notebook 07 run. These files provided the comparison without loading B's model
weights or training B again. The checks confirmed that C used matching data,
model, starting values, and training settings apart from the learning rate.
The default option below used this bundled reference.

In [3]:
REFERENCE_METHOD = "Bundled verified B metadata" #@param ["Bundled verified B metadata", "Upload B results ZIP", "Existing B runtime folder", "C only"]
B_REFERENCE = None
if REFERENCE_METHOD == "Bundled verified B metadata":
    B_REFERENCE = protocol.read_b_directory(C_WORK/"reference_B")
    B_REFERENCE["source_archive"] = json.loads((C_WORK/"reference_B/source.json").read_text())
elif REFERENCE_METHOD == "Upload B results ZIP":
    b_upload = files.upload()
    assert len(b_upload)==1, "Choose exactly the completed 07 results ZIP."
    b_name = next(iter(b_upload))
    B_REFERENCE = protocol.read_b_zip(b_upload[b_name], source=b_name)
    del b_upload
elif REFERENCE_METHOD == "Existing B runtime folder":
    B_DIRECTORY = "/content/music_validation_only_1n3xkmod/validation_only_results/B_train_weighted_ABC" #@param {type:"string"}
    B_REFERENCE = protocol.read_b_directory(B_DIRECTORY)
else:
    assert REFERENCE_METHOD == "C only"
    print("No B reference: C will have standalone validation results only.")
if B_REFERENCE is not None:
    contract.save_json(C_OUTPUT/"B_reference.json", B_REFERENCE)
    print("Actual B validation accuracy:", B_REFERENCE["summary"]["selected"]["validation"]["accuracy"])
    print("Actual B validation macro-F1:", B_REFERENCE["summary"]["selected"]["validation"]["macro_f1"])

Actual B validation accuracy: 0.5757575757575758
Actual B validation macro-F1: 0.464881371858116


## 4. Experiment C: A Smaller Learning Rate

The learning rate controlled the size of each training update. C lowered it
from **0.0002 to 0.0001** to test whether smaller updates improved classification.

C started with the same pretrained model and a fresh LoRA adapter. It trained
for **three epochs**, or three passes through the training data. Class weights,
the random seed, batching, and the other training settings matched B. This
kept the comparison focused on the lower learning rate.

The run saved a model checkpoint after each epoch. A checkpoint is a saved
version of the trained adapter.

In [4]:
import torch, importlib.metadata
assert torch.cuda.is_available() and "T4" in torch.cuda.get_device_name(0)
assert torch.__version__.split("+")[0] == "2.11.0"
from transformers import AutoTokenizer
import c_training as training
tokenizer = AutoTokenizer.from_pretrained(training.MODEL_ID, revision=training.MODEL_REVISION, trust_remote_code=False)
if tokenizer.pad_token_id is None: tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
VERSIONS = {name:importlib.metadata.version(name) for name in ["torch",*PINS]}
contract.save_json(C_OUTPUT/"protocol.json", {
    "experiment":protocol.C_NAME, "learning_rate":protocol.C_LEARNING_RATE,
    "only_training_setting_changed_from_B":"learning_rate: 2e-4 -> 1e-4",
    "bundle_sha256":EXPECTED_BUNDLE_SHA256, "versions":VERSIONS,
    "input_sha256":manifest["csv_sha256"], "b_reference_supplied":B_REFERENCE is not None,
    "diagnostics":"post-training checkpoint reload/scoring only",
    "selection":"validation macro-F1, lower unweighted validation CE, earlier epoch",
    "independent_human_evaluation":False, "confirmatory_final_comparison_performed":False})
try:
    summary_C = training.run_experiment(protocol.C_NAME, tokenizer, parts["train"], parts["validation"],
        manifest, C_OUTPUT, VERSIONS, learning_rate=protocol.C_LEARNING_RATE, b_reference=B_REFERENCE)
except Exception as error:
    protocol.save_failure(C_OUTPUT, "training", error)
    raise
print("C training finished. Reload diagnostics still required.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

C_train_weighted_ABC_lr1e4 epoch 1 validation macro-F1 0.150794 predictions {'specific_musical_observation': 1, 'general_opinion': 28, 'background_other': 4} updates/skips 15 3
C_train_weighted_ABC_lr1e4 epoch 2 validation macro-F1 0.312292 predictions {'specific_musical_observation': 25, 'general_opinion': 2, 'background_other': 6} updates/skips 18 0
C_train_weighted_ABC_lr1e4 epoch 3 validation macro-F1 0.423016 predictions {'specific_musical_observation': 22, 'general_opinion': 5, 'background_other': 6} updates/skips 18 0
C training finished. Reload diagnostics still required.


## 5. Checking the Saved Models

After C finished training, each of its three checkpoints was reloaded and
scored on the training and validation passages. The reloaded validation
predictions matched the predictions saved during training.

The best checkpoint was selected using validation **macro-F1**, which gives
each category equal importance. Ties used lower validation loss, then the
earlier epoch. Training scores helped describe model fit; they were not used
to choose the checkpoint.

The saved diagnostics included per-category scores, confusion matrices,
predictions, and errors. These checks used the saved models without further
training.

In [5]:
import c_diagnostics
try:
    DIAGNOSTICS = c_diagnostics.run_post_training_diagnostics(C_OUTPUT/protocol.C_NAME, tokenizer, parts, summary_C)
    summary_C["post_training_diagnostics_complete"] = True
    summary_C["saved_reloaded_selected_adapter_agreement"] = DIAGNOSTICS["saved_reloaded_selected_adapter_agreement"]
    contract.save_json(C_OUTPUT/"C_completed_summary.json", summary_C)
    if B_REFERENCE is not None:
        comparison = protocol.compare_with_b(B_REFERENCE, summary_C)
        contract.save_json(C_OUTPUT/"B_C_validation_comparison.json", comparison)
    contract.save_json(C_OUTPUT/"completion.json", {
        "status":"C_completed_with_reload_checks", "selected_epoch":summary_C["selected"]["epoch"],
        "diagnostics":"PASS", "saved_reloaded_selected_adapter_agreement":"PASS",
        "b_comparison_available":B_REFERENCE is not None, "independent_human_evaluation":False,
        "confirmatory_final_comparison_performed":False})
except Exception as error:
    protocol.save_failure(C_OUTPUT, "diagnostics", error)
    raise
import pandas as pd
rows=[]
for report in DIAGNOSTICS["epochs"]:
    for split in ["train","validation"]:
        m=report[split]
        rows.append({"epoch":report["epoch"],"split":split,"accuracy":m["accuracy"],
            "macro_f1":m["macro_f1"],"unweighted_ce":m["unweighted_ce"],"prediction_counts":m["prediction_counts"]})
display(pd.DataFrame(rows))
selected=DIAGNOSTICS["selected"]
print("Selected epoch (validation criterion):",selected["epoch"])
display(pd.DataFrame(selected["validation"]["confusion_matrix"], index=contract.LABELS, columns=contract.LABELS))
display(pd.DataFrame(selected["validation"]["per_class"]).T)
print("Confusion matrix rows = labeled class; columns = predicted class.")
if B_REFERENCE is not None:
    print("Actual C minus B accuracy:",comparison["delta_accuracy"])
    print("Actual C minus B macro-F1:",comparison["delta_macro_f1"])
else:
    print("B files unavailable: no C-versus-B claim is made.")

Reloaded epoch 1 | train accuracy 0.257143 | validation accuracy 0.181818 | validation macro-F1 0.150794
Reloaded epoch 2 | train accuracy 0.564286 | validation accuracy 0.484848 | validation macro-F1 0.312292
Reloaded epoch 3 | train accuracy 0.571429 | validation accuracy 0.515152 | validation macro-F1 0.423016


,epoch,split,accuracy,macro_f1,unweighted_ce,prediction_counts
0,1,train,0.257143,0.181195,1.199223,"{'specific_musical_observation': 3, 'general_o..."
1,1,validation,0.181818,0.150794,1.257334,"{'specific_musical_observation': 1, 'general_o..."
2,2,train,0.564286,0.368889,1.011755,"{'specific_musical_observation': 118, 'general..."
3,2,validation,0.484848,0.312292,1.032220,"{'specific_musical_observation': 25, 'general_..."
4,3,train,0.571429,0.462267,1.012106,"{'specific_musical_observation': 89, 'general_..."
5,3,validation,0.515152,0.423016,1.047831,"{'specific_musical_observation': 22, 'general_..."


Selected epoch (validation criterion): 3


,specific_musical_observation,general_opinion,background_other
specific_musical_observation,13,3,2
general_opinion,3,2,2
background_other,6,0,2


,precision,recall,f1,support
specific_musical_observation,0.590909,0.722222,0.650000,18.0
general_opinion,0.400000,0.285714,0.333333,7.0
background_other,0.333333,0.250000,0.285714,8.0


Confusion matrix rows = labeled class; columns = predicted class.
Actual C minus B accuracy: -0.06060606060606066
Actual C minus B macro-F1: -0.041865498842243


## 6. Saving Experiment C

C completed the checkpoint checks and produced
`music_validation_only_C_lr1e4_COMPLETE_results.zip`. The archive preserved
the saved adapters, predictions, scores, training logs, and diagnostic results.

The export cell created the archive in the Colab runtime. The completed
archive and this executed notebook were retained as the record of the run.

In [6]:
result_archive, archive_state = protocol.export_results(C_OUTPUT, C_WORK)
print("C archive status:", archive_state)
print("Download from Colab Files:", result_archive)
print("Also choose File > Download > Download .ipynb for this executed notebook.")

C archive status: COMPLETE
Download from Colab Files: /content/music_followup_C_kkipo9qx/music_validation_only_C_lr1e4_COMPLETE_results.zip
Also choose File > Download > Download .ipynb for this executed notebook.


## 7. Experiment C Results

C's selected checkpoint was **epoch 3**. It correctly classified **17 of 33**
validation passages, with accuracy **0.515152** and macro-F1 **0.423016**.
B correctly classified **19 of 33**, with macro-F1 **0.464881**.

The smaller learning rate did not improve C's validation results over B.
Experiment D then tested a longer training run at the same lower learning rate.

## 8. Experiment D: More Training Epochs

D started with a fresh pretrained model and a fresh LoRA adapter. It was
**not a continuation of C**. It used learning rate **0.0001** and trained for
**five epochs** instead of three.

The longer run increased planned updates from **54 to 90** and warmup updates
from **5 to 9**. Warmup gradually increased the learning rate at the start of
training. The later learning-rate decay also lasted longer, so D changed both
the training budget and the schedule. The data, class weights, model, and
checkpoint-selection method stayed the same.

### 8.1. Checking the Saved Checkpoints

A diagnostic helper needed a repair before it could finish checking D's saved
models. The repair restored the helper's missing function references. It did
not retrain the model or replace the checkpoints. All five checkpoints were
then checked, and the reloaded validation predictions matched the saved results.
The repair cell appears between D's training and diagnostic cells below.

### 8.2. Comparing B, C, and D

| Experiment | Selected epoch | Correct validation predictions | Accuracy | Macro-F1 |
|---|---:|---:|---:|---:|
| B: weighted, 3 epochs, learning rate 0.0002 | 2 | 19 / 33 | 0.575758 | 0.464881 |
| C: weighted, 3 epochs, learning rate 0.0001 | 3 | 17 / 33 | 0.515152 | 0.423016 |
| D: weighted, 5 epochs, learning rate 0.0001 | 5 | 19 / 33 | 0.575758 | 0.461871 |

B remained the best model by validation macro-F1. D matched B's accuracy,
but its macro-F1 was lower by about **0.0030**. The longer run therefore did
not improve the selection score on this validation set.

### 8.3. Interpretation and Limitations

The same 33 validation passages were reused for checkpoint selection and
experiment comparisons. These were development results, not an independent
final test. A fresh, independently human-labeled evaluation set would be
needed to assess performance on new passages. The small difference between
B and D does not establish a significant improvement in generalization.

### References

- [Saved Experiment B summary](../../results/evidence/B_summary.json)
- [Saved Experiment C summary](../../results/evidence/C_summary.json)
- [Saved Experiment D summary](../../results/evidence/D_summary.json)
- [Data and reproduction notes](../../docs/REPRODUCTION.md)

In [7]:
"""Build private D functions from the exact reviewed C source; never mutate C.

No ML imports, model execution, uploads or training happen in this factory.
The structural edits are deliberately narrow and fail if the source differs.
"""
import ast
import types

D_NAME = 'D_train_weighted_ABC_lr1e4_5epochs'


def make_private_D(training, diagnostics, protocol, contract, training_source, diagnostic_source):
    def require(value, message):
        if not value: raise ValueError(message)

    def check_d_request(name, learning_rate, manifest, epochs):
        require(name == D_NAME and epochs == 5, 'Exactly one fresh five-epoch D configuration is allowed.')
        # Reuse C's unchanged immutable-data, seed and 1e-4 learning-rate checks.
        protocol.check_c_request(protocol.C_NAME, learning_rate, manifest)

    def verify_d_settings(b, d):
        require(b['epochs'] == 3 and d['epochs'] == 5, 'Expected B=3 epochs and D=5 epochs.')
        # All other settings remain checked by the original strict validator.
        # This copy affects comparison metadata only, never the actual D configuration.
        protocol.verify_controlled_settings(b, dict(d, epochs=3))

    private_protocol = types.SimpleNamespace(C_NAME=D_NAME, C_LEARNING_RATE=protocol.C_LEARNING_RATE,
        check_d_request=check_d_request, verify_initialization=protocol.verify_initialization,
        verify_controlled_settings=verify_d_settings)

    def get_function(source, name):
        matches=[n for n in ast.parse(source).body if isinstance(n, ast.FunctionDef) and n.name == name]
        require(len(matches) == 1, 'Expected exactly one reviewed function: '+name)
        return matches[0]

    def add_epochs_parameter(function):
        require(not any(a.arg == 'epochs' for a in function.args.kwonlyargs), 'Unexpected existing epoch argument.')
        function.args.kwonlyargs.append(ast.arg(arg='epochs'))
        function.args.kw_defaults.append(ast.Constant(value=5))

    run = get_function(training_source, 'run_experiment')
    add_epochs_parameter(run)
    first = run.body[0]
    require(isinstance(first, ast.Expr) and isinstance(first.value, ast.Call)
        and isinstance(first.value.func, ast.Attribute) and first.value.func.attr == 'check_c_request',
        'C entry guard differs from reviewed source.')
    first.value.func.attr = 'check_d_request'; first.value.args.append(ast.Name(id='epochs', ctx=ast.Load()))
    edits=[]
    for node in run.body:
        if isinstance(node, ast.Assign) and any(isinstance(t, ast.Name) and t.id == 'settings' for t in node.targets):
            require(isinstance(node.value, ast.Dict), 'Unexpected settings schema.')
            for index,key in enumerate(node.value.keys):
                if isinstance(key, ast.Constant) and key.value == 'epochs':
                    require(isinstance(node.value.values[index],ast.Constant) and node.value.values[index].value == 3,
                        'Expected original three-epoch setting.')
                    node.value.values[index] = ast.Name(id='epochs',ctx=ast.Load()); edits.append('settings')
        if isinstance(node, ast.Assign) and any(isinstance(t, ast.Name) and t.id == 'planned_steps' for t in node.targets):
            require(isinstance(node.value,ast.BinOp) and isinstance(node.value.left,ast.Constant)
                and node.value.left.value == 3, 'Unexpected original schedule budget.')
            node.value.left = ast.Name(id='epochs',ctx=ast.Load()); edits.append('schedule')
        if isinstance(node, ast.For) and isinstance(node.target,ast.Name) and node.target.id == 'epoch':
            require(ast.dump(node.iter) == ast.dump(ast.parse('range(1, 4)',mode='eval').body),
                    'Unexpected original epoch loop.')
            node.iter = ast.parse('range(1, epochs + 1)',mode='eval').body; edits.append('loop')
    require(sorted(edits) == ['loop','schedule','settings'], 'Unexpected epoch edit count.')
    train_module = types.ModuleType('d_private_training')
    train_module.__dict__.update(training.__dict__)
    train_module.__dict__.update(__name__='d_private_training', protocol=private_protocol, EXPERIMENTS={D_NAME:True})
    train_ast = ast.fix_missing_locations(ast.Module(body=[run],type_ignores=[]))
    exec(compile(train_ast,'<D private training: reviewed C plus explicit epochs parameter>','exec'),train_module.__dict__)

    diagnose = get_function(diagnostic_source,'run_post_training_diagnostics')
    add_epochs_parameter(diagnose)
    diagnose.body.insert(0,ast.parse('assert epochs == 5, "D diagnostics require exactly five completed epochs."').body[0])
    class DiagnosticEdits(ast.NodeTransformer):
        def __init__(self): self.edits=[]
        def visit_Compare(self,node):
            self.generic_visit(node)
            if (len(node.comparators)==1 and isinstance(node.comparators[0],ast.List)
                and ast.dump(node.comparators[0]) == ast.dump(ast.parse('[1,2,3]',mode='eval').body)):
                node.comparators[0] = ast.parse('list(range(1, epochs + 1))',mode='eval').body
                self.edits.append('history')
            return node
        def visit_Constant(self,node):
            replacements={'all_three_checkpoints_reloaded':'all_requested_checkpoints_reloaded',
                          'Expected exactly three completed epochs.':'Expected exactly five completed epochs.'}
            if isinstance(node.value,str) and node.value in replacements:
                self.edits.append(node.value); node.value=replacements[node.value]
            return node
    change=DiagnosticEdits(); diagnose=change.visit(diagnose)
    require(sorted(change.edits)==sorted(['history','all_three_checkpoints_reloaded',
        'Expected exactly three completed epochs.']), 'Unexpected checkpoint diagnostic edit count.')
    diagnostic_module = types.ModuleType('d_private_diagnostics')
    diagnostic_module.__dict__.update(diagnostics.__dict__)
    diagnostic_module.__dict__.update(__name__='d_private_diagnostics',protocol=private_protocol,training=train_module)
    diagnostic_ast=ast.fix_missing_locations(ast.Module(body=[diagnose],type_ignores=[]))
    exec(compile(diagnostic_ast,'<D private post-training reload diagnostics>','exec'),diagnostic_module.__dict__)
    return train_module, diagnostic_module, ast.unparse(train_ast), ast.unparse(diagnostic_ast)

# D1 — preflight and private D functions. Run only after preserving C.
from pathlib import Path
import hashlib, json, math
assert "C_WORK" in globals() and "C_OUTPUT" in globals(), "Use the existing completed notebook 08 runtime."
assert (C_OUTPUT/"completion.json").is_file(), "Finish C diagnostics first."
assert json.loads((C_OUTPUT/"completion.json").read_text())["status"] == "C_completed_with_reload_checks"
assert (C_WORK/"music_validation_only_C_lr1e4_COMPLETE_results.zip").is_file(), "Run C section 6, then download C ZIP and executed 08 before D."
import torch
assert torch.cuda.is_available() and "T4" in torch.cuda.get_device_name(0), "Keep the existing working T4. Do not allocate/restart a runtime here."
import c_training as _D_C_training
import c_diagnostics as _D_C_diagnostics
import c_protocol as _D_C_protocol
import c_contract as _D_C_contract
_D_EXPECTED_HASHES = {'c_training.py': 'b4c3ba63af0926ff337fa7acfdc21a5e7126f903b47d96c41bdf91e5dfb1d284', 'c_diagnostics.py': 'b15fb3ff12726611bcd8f7ab8067cc354283a6b98dbe35039512b79960352a44', 'c_protocol.py': '484bd6562e63afed59bb98d3d3246e8419354598cb651cf47f9bc55c2d3c1db2', 'c_contract.py': 'e663fc148bee844f085cc5e644cf5a1f01ab3e6c66d208f19b2a96afa0ee76cb'}
for _D_name,_D_digest in _D_EXPECTED_HASHES.items():
    assert hashlib.sha256((C_WORK/_D_name).read_bytes()).hexdigest() == _D_digest, "Reviewed C source changed: "+_D_name
D_PARTS,D_MANIFEST = _D_C_contract.load_data(C_WORK)
D_B_REFERENCE = _D_C_protocol.read_b_directory(C_WORK/"reference_B")
D_C_REFERENCE = json.loads((C_OUTPUT/"C_completed_summary.json").read_text())
assert D_C_REFERENCE["post_training_diagnostics_complete"] is True
D_TRAINING,D_DIAGNOSTICS,D_TRAIN_SOURCE,D_DIAGNOSTIC_SOURCE = make_private_D(
    _D_C_training,_D_C_diagnostics,_D_C_protocol,_D_C_contract,
    (C_WORK/"c_training.py").read_text(),(C_WORK/"c_diagnostics.py").read_text())
D_WORK = C_WORK
D_OUTPUT = D_WORK/"D_five_epoch_results"
D_OUTPUT.mkdir(exist_ok=False)  # One D attempt; never overwrite or repeat implicitly.
with (D_OUTPUT/"D_training_function.py").open("x") as f: f.write(D_TRAIN_SOURCE+"\n")
with (D_OUTPUT/"D_diagnostic_function.py").open("x") as f: f.write(D_DIAGNOSTIC_SOURCE+"\n")
_D_C_contract.save_json(D_OUTPUT/"B_reference.json",D_B_REFERENCE)
_D_C_contract.save_json(D_OUTPUT/"C_reference_summary.json",D_C_REFERENCE)
_D_C_contract.save_json(D_OUTPUT/"D_protocol.json",{
    "experiment":D_NAME,"start":"fresh pretrained base and fresh adapter; not a resume",
    "learning_rate":1e-4,"epochs":5,"planned_optimizer_steps":90,"warmup_steps":9,
    "schedule":"linear warmup and decay; advance only on successful updates",
    "changes_from_C":{"epoch_budget":"3 -> 5","planned_steps":"54 -> 90","warmup_steps":"5 -> 9","decay_horizon":"54 -> 90"},
    "unchanged":"model/revision, seed, inputs/splits/labels, prompt, training-only weights, LoRA, batch, accumulation, optimizer, dropout, selection criterion",
    "source_hashes":_D_EXPECTED_HASHES,"input_sha256":D_MANIFEST["csv_sha256"],
    "private_namespace_only":True,"C_helpers_modified":False,"old_test_accessed":False,
    "selection":"validation macro-F1, then lower unweighted validation CE, then earlier epoch",
    "independent_human_evaluation":False,"confirmatory_final_comparison_performed":False})
print("D preflight passed. New output:",D_OUTPUT)
print("One fresh 5-epoch trial; 90 planned steps, 9 warmup steps. C remains unchanged.")



D preflight passed. New output: /content/music_followup_C_kkipo9qx/D_five_epoch_results
One fresh 5-epoch trial; 90 planned steps, 9 warmup steps. C remains unchanged.


In [8]:
# D2 — train D ONCE. Do not rerun any original C cell.
try:
    D_SUMMARY = D_TRAINING.run_experiment(D_NAME,tokenizer,D_PARTS["train"],D_PARTS["validation"],
        D_MANIFEST,D_OUTPUT,VERSIONS,learning_rate=1e-4,epochs=5,b_reference=D_B_REFERENCE)
except Exception as D_error:
    _D_C_protocol.save_failure(D_OUTPUT,"D_training",D_error)
    raise
print("D training complete. Run D3 reload diagnostics next.")


D_train_weighted_ABC_lr1e4_5epochs epoch 1 validation macro-F1 0.148148 predictions {'specific_musical_observation': 0, 'general_opinion': 29, 'background_other': 4} updates/skips 15 3
D_train_weighted_ABC_lr1e4_5epochs epoch 2 validation macro-F1 0.42717 predictions {'specific_musical_observation': 28, 'general_opinion': 4, 'background_other': 1} updates/skips 18 0
D_train_weighted_ABC_lr1e4_5epochs epoch 3 validation macro-F1 0.254377 predictions {'specific_musical_observation': 5, 'general_opinion': 10, 'background_other': 18} updates/skips 18 0
D_train_weighted_ABC_lr1e4_5epochs epoch 4 validation macro-F1 0.392498 predictions {'specific_musical_observation': 19, 'general_opinion': 6, 'background_other': 8} updates/skips 18 0
D_train_weighted_ABC_lr1e4_5epochs epoch 5 validation macro-F1 0.461871 predictions {'specific_musical_observation': 24, 'general_opinion': 4, 'background_other': 5} updates/skips 18 0
D training complete. Run D3 reload diagnostics next.


In [10]:
# D3 REPAIR ONCE — preparation only; then rerun D3 below. Never rerun D1/D2.
# Fix two omitted diagnostic helpers, preserve the failed inference attempt,
# and leave all trained adapters, training results, C helpers and C outputs intact.
import ast, builtins, dis, gc, hashlib, json, sys, traceback, types
from pathlib import Path

def _D3_prepare_repair():
    assert D_NAME == 'D_train_weighted_ABC_lr1e4_5epochs'
    assert D_OUTPUT == C_WORK/'D_five_epoch_results'
    assert D_SUMMARY['experiment'] == D_NAME
    assert D_SUMMARY['status'] == 'completed_validation_only'
    assert D_SUMMARY['frozen_base_verified'] is True
    assert set(D_PARTS) == {'train', 'validation'}
    for name, digest in _D_EXPECTED_HASHES.items():
        assert hashlib.sha256((C_WORK/name).read_bytes()).hexdigest() == digest, name
    experiment = D_OUTPUT/D_NAME
    history = json.loads((experiment/'training_history.json').read_text())
    assert [r['epoch'] for r in history] == [1, 2, 3, 4, 5]
    for epoch in range(1, 6):
        assert (experiment/f'adapter_epoch_{epoch}'/'adapter_model.safetensors').is_file()
    assert not any((D_OUTPUT/name).exists() for name in (
        'completion.json', 'D_completed_summary.json', 'D_vs_B_and_C_validation.json', 'D3_repair.json'))
    partial = experiment/'post_training_diagnostics'
    failure = D_OUTPUT/'failure_D_diagnostics.json'
    assert partial.is_dir() and failure.is_file(), 'Expected the preserved failed D3 attempt.'
    failure_record = json.loads(failure.read_text())
    assert failure_record['error_type'] == 'AttributeError' and 'save_errors' in failure_record['message']
    assert not (partial/'diagnostic_summary.json').exists(), 'Do not replace completed diagnostics.'
    partial_files = [p for p in partial.rglob('*') if p.is_file()]
    assert partial_files and all(not p.is_symlink() for p in partial.rglob('*'))
    assert set(str(p.relative_to(partial)) for p in partial_files) <= {
        'tokenization_audit.csv', 'epoch_1/train_predictions.csv'}

    # Audit EVERY protocol/training/contract attribute used by the reviewed function.
    function = D_DIAGNOSTICS.run_post_training_diagnostics
    assert function.__globals__ is D_DIAGNOSTICS.__dict__
    old_protocol = D_DIAGNOSTICS.protocol
    assert old_protocol is D_TRAINING.protocol
    patched_protocol = types.SimpleNamespace(**vars(old_protocol))
    for name in ('save_errors', 'compare_prediction_files'):
        assert not hasattr(old_protocol, name), 'This repair has already been applied.'
        setattr(patched_protocol, name, getattr(_D_C_protocol, name))
    tree = ast.parse(D_DIAGNOSTIC_SOURCE)
    dependencies = {}
    for root, module in [('protocol', patched_protocol), ('training', D_DIAGNOSTICS.training),
                         ('contract', D_DIAGNOSTICS.contract)]:
        names = sorted({n.attr for n in ast.walk(tree) if isinstance(n, ast.Attribute)
                        and isinstance(n.value, ast.Name) and n.value.id == root})
        assert all(hasattr(module, name) for name in names), (root, names)
        dependencies[root] = names
    assert dependencies['protocol'] == ['C_NAME', 'compare_prediction_files', 'save_errors']
    for instruction in dis.get_instructions(function):
        if instruction.opname == 'LOAD_GLOBAL':
            name = instruction.argval
            assert name in function.__globals__ or hasattr(builtins, name), name

    def hashes(files, root):
        return {str(p.relative_to(root)): hashlib.sha256(p.read_bytes()).hexdigest() for p in files}
    protected = [p for p in experiment.rglob('*') if p.is_file() and partial not in p.parents]
    protected += [D_OUTPUT/name for name in ('D_training_function.py', 'D_diagnostic_function.py',
                                           'D_protocol.json', 'B_reference.json', 'C_reference_summary.json')]
    assert all(not p.is_symlink() for p in protected)
    before = hashes(protected, D_OUTPUT)
    partial_hashes = hashes(partial_files, partial)
    failure_hash = hashlib.sha256(failure.read_bytes()).hexdigest()
    archive = D_OUTPUT/'diagnostic_failures'/'attempt_1'
    assert not archive.exists(), 'Preserve the previous repair; do not repeat implicitly.'
    archive.mkdir(parents=True, exist_ok=False)
    partial.rename(archive/'post_training_diagnostics')
    failure.rename(archive/failure.name)
    assert hashes([archive/'post_training_diagnostics'/p for p in partial_hashes],
                  archive/'post_training_diagnostics') == partial_hashes
    assert hashlib.sha256((archive/failure.name).read_bytes()).hexdigest() == failure_hash
    assert hashes(protected, D_OUTPUT) == before
    # Only the private diagnostic module changes; training and C namespaces do not.
    D_DIAGNOSTICS.protocol = patched_protocol
    assert D_TRAINING.protocol is old_protocol
    assert D_DIAGNOSTICS.protocol.save_errors is _D_C_protocol.save_errors
    assert D_DIAGNOSTICS.protocol.compare_prediction_files is _D_C_protocol.compare_prediction_files
    _D_C_contract.save_json(D_OUTPUT/'D3_repair.json', {
        'status': 'PREPARED_NOT_DIAGNOSTIC_COMPLETION', 'bindings_added': ['save_errors', 'compare_prediction_files'],
        'preserved_failure_directory': str(archive), 'failed_diagnostic_files_sha256': partial_hashes,
        'failure_record_sha256': failure_hash, 'protected_training_files_sha256': before,
        'dependencies_checked': dependencies, 'retraining_performed': False,
        'reload_checks_and_selection_unchanged': True})
    # Release model locals held by the failed diagnostic traceback, after preserving it on disk.
    tb = getattr(sys, 'last_traceback', None)
    cursor = tb
    while cursor is not None:
        if cursor.tb_frame.f_code is function.__code__:
            traceback.clear_frames(tb)
            break
        cursor = cursor.tb_next
    gc.collect()
    D_DIAGNOSTICS.torch.cuda.empty_cache()
    print('D3 repair prepared. Failed inference evidence preserved at:', archive)
    print('Training/checkpoints unchanged. Run D3 below once; do not run D1 or D2.')

_D3_prepare_repair()


D3 repair prepared. Failed inference evidence preserved at: /content/music_followup_C_kkipo9qx/D_five_epoch_results/diagnostic_failures/attempt_1
Training/checkpoints unchanged. Run D3 below once; do not run D1 or D2.


In [11]:
# D3 — inference only, after all five epochs; select by validation only.
try:
    D_REPORT = D_DIAGNOSTICS.run_post_training_diagnostics(D_OUTPUT/D_NAME,tokenizer,D_PARTS,D_SUMMARY,epochs=5)
    D_SUMMARY["post_training_diagnostics_complete"] = True
    D_SUMMARY["saved_reloaded_selected_adapter_agreement"] = D_REPORT["saved_reloaded_selected_adapter_agreement"]
    _D_C_contract.save_json(D_OUTPUT/"D_completed_summary.json",D_SUMMARY)
    D_COMPARISON = []
    for D_label,D_result in [("B (3 epochs, LR2e-4)",D_B_REFERENCE["summary"]),
                             ("C (3 epochs, LR1e-4)",D_C_REFERENCE),("D (5 epochs, LR1e-4)",D_SUMMARY)]:
        D_m=D_result["selected"]["validation"]
        D_COMPARISON.append({"experiment":D_label,"selected_epoch":D_result["selected"]["epoch"],
            "accuracy":D_m["accuracy"],"macro_f1":D_m["macro_f1"],"prediction_counts":D_m["prediction_counts"]})
    _D_C_contract.save_json(D_OUTPUT/"D_vs_B_and_C_validation.json",D_COMPARISON)
    _D_C_contract.save_json(D_OUTPUT/"completion.json",{
        "status":"D_completed_with_reload_checks","selected_epoch":D_SUMMARY["selected"]["epoch"],
        "all_five_checkpoints_reloaded":True,"saved_reloaded_selected_adapter_agreement":"PASS",
        "training_budget_and_schedule_changed":True,"independent_human_evaluation":False,
        "confirmatory_final_comparison_performed":False})
except Exception as D_error:
    _D_C_protocol.save_failure(D_OUTPUT,"D_diagnostics",D_error)
    raise
import pandas as pd
display(pd.DataFrame(D_COMPARISON))
print("Keep B if D does not beat its validation macro-F1. No independent final evaluation is claimed.")


Reloaded epoch 1 | train accuracy 0.257143 | validation accuracy 0.181818 | validation macro-F1 0.148148
Reloaded epoch 2 | train accuracy 0.564286 | validation accuracy 0.575758 | validation macro-F1 0.42717
Reloaded epoch 3 | train accuracy 0.371429 | validation accuracy 0.272727 | validation macro-F1 0.254377
Reloaded epoch 4 | train accuracy 0.607143 | validation accuracy 0.484848 | validation macro-F1 0.392498
Reloaded epoch 5 | train accuracy 0.614286 | validation accuracy 0.575758 | validation macro-F1 0.461871


,experiment,selected_epoch,accuracy,macro_f1,prediction_counts
0,"B (3 epochs, LR2e-4)",2,0.575758,0.464881,"{'specific_musical_observation': 25, 'general_..."
1,"C (3 epochs, LR1e-4)",3,0.515152,0.423016,"{'specific_musical_observation': 22, 'general_..."
2,"D (5 epochs, LR1e-4)",5,0.575758,0.461871,"{'specific_musical_observation': 24, 'general_..."


Keep B if D does not beat its validation macro-F1. No independent final evaluation is claimed.


In [12]:
# D4 — export D separately, including an incomplete attempt if needed.
import zipfile
D_COMPLETE_FILE=D_OUTPUT/"completion.json"
D_ARCHIVE_STATE="COMPLETE" if (D_COMPLETE_FILE.is_file() and
    json.loads(D_COMPLETE_FILE.read_text()).get("status")=="D_completed_with_reload_checks") else "INCOMPLETE"
D_ARCHIVE=D_WORK/("music_validation_only_D_five_epoch_"+D_ARCHIVE_STATE+"_results.zip")
D_MEMBERS=sorted(p for p in D_OUTPUT.rglob("*") if p.is_file())
assert D_MEMBERS and all(not p.is_symlink() for p in D_MEMBERS)
D_HASHES={str(p.relative_to(D_OUTPUT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in D_MEMBERS}
with zipfile.ZipFile(D_ARCHIVE,"x",zipfile.ZIP_DEFLATED) as D_zip:
    for D_file in D_MEMBERS: D_zip.write(D_file,str(D_file.relative_to(D_OUTPUT)))
    D_zip.writestr("ARCHIVE_SHA256.json",json.dumps(D_HASHES,indent=2))
print("Download D archive from Files:",D_ARCHIVE)
print("Also download the executed SAME notebook 08. Preserve B and C. Stop after this one D trial.")


Download D archive from Files: /content/music_followup_C_kkipo9qx/music_validation_only_D_five_epoch_COMPLETE_results.zip
Also download the executed SAME notebook 08. Preserve B and C. Stop after this one D trial.
